# YOLO26s Classifier Training on Roboflow

AdamW + cosine LR, early stopping, periodic checkpoints, and metrics export.

In [20]:
# %pip install ultralytics roboflow

In [21]:
from __future__ import annotations
import json, os, shutil
from datetime import datetime
from pathlib import Path
from typing import Any
from ultralytics import YOLO

In [22]:
MODEL_NAME='yolo26s-cls.pt'
EPOCHS=100
IMGSZ=224
BATCH=32
WORKERS=8
DEVICE=0
OPTIMIZER='AdamW'
LR0=1e-3
LRF=1e-2
WEIGHT_DECAY=5e-4
COS_LR=True
PATIENCE=20
SAVE_PERIOD=5
OUTPUT_ROOT=Path('runs')/'yolo26s_cls_roboflow'
RUN_NAME=f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# Set this to the local dataset folder (containing train/valid subfolders).
# Set to None to download from Roboflow instead.
DATASET_DIR=r'C:\BME\MSC\4. félév\GL\GLHF\datasets\classification_cls'
ROBOFLOW_API_KEY=os.getenv('ROBOFLOW_API_KEY', None)
ROBOFLOW_WORKSPACE='gl-hzi'
ROBOFLOW_PROJECT='plate-detect-8tgbr'
ROBOFLOW_VERSION=2
ROBOFLOW_FORMAT='yolo26'
DATASET_CACHE_DIR=Path.home()/'datasets'/'roboflow'
FORCE_DOWNLOAD=False

In [23]:
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path): return str(obj)
    if hasattr(obj, 'item'):
        try: return obj.item()
        except Exception: return str(obj)
    return obj

def resolve_dataset_dir(dataset_dir, api_key, workspace, project, version, export_format='yolo26',
                        dataset_cache_dir=Path.home()/'datasets'/'roboflow', force_download=False):
    if dataset_dir is not None:
        p = Path(dataset_dir).resolve()
        if not p.exists(): raise FileNotFoundError(f'Dataset dir not found: {p}')
        print(f'Using local dataset: {p}')
        return p
    missing = []
    if not api_key: missing.append('ROBOFLOW_API_KEY')
    if not workspace: missing.append('ROBOFLOW_WORKSPACE')
    if not project: missing.append('ROBOFLOW_PROJECT')
    if version is None: missing.append('ROBOFLOW_VERSION')
    if missing: raise ValueError('Missing dataset config: ' + ', '.join(missing))
    from roboflow import Roboflow
    target_dir = (dataset_cache_dir / workspace / project / f'v{version}').resolve()
    if target_dir.exists() and any(target_dir.iterdir()) and not force_download:
        print(f'Using cached dataset: {target_dir}')
        return target_dir
    target_dir.mkdir(parents=True, exist_ok=True)
    rf = Roboflow(api_key=api_key)
    ds = rf.workspace(workspace).project(project).version(version).download(export_format, location=str(target_dir))
    return Path(ds.location).resolve()

def export_metrics(run_dir: Path, train_result: Any, val_result: Any) -> Path:
    m = run_dir / 'metrics'; m.mkdir(parents=True, exist_ok=True)
    summary = {
      'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
      'run_dir': str(run_dir),
      'train_result': to_jsonable(getattr(train_result, 'results_dict', {})),
      'val_result': to_jsonable(getattr(val_result, 'results_dict', {})),
      'best_checkpoint': str(run_dir / 'weights' / 'best.pt'),
      'last_checkpoint': str(run_dir / 'weights' / 'last.pt')
    }
    (m / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    for a in ['results.csv', 'results.png', 'args.yaml']:
        s = run_dir / a
        if s.exists(): shutil.copy2(s, m / s.name)
    return m

In [24]:
dataset_dir = resolve_dataset_dir(
    dataset_dir=DATASET_DIR,
    api_key=ROBOFLOW_API_KEY,
    workspace=ROBOFLOW_WORKSPACE,
    project=ROBOFLOW_PROJECT,
    version=ROBOFLOW_VERSION,
    export_format=ROBOFLOW_FORMAT,
    dataset_cache_dir=DATASET_CACHE_DIR,
    force_download=FORCE_DOWNLOAD,
)
print('Dataset dir:', dataset_dir)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_NAME)
train_kwargs = {
    'data': str(dataset_dir), 'task': 'classify',
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH,
    'workers': WORKERS, 'device': DEVICE, 'optimizer': OPTIMIZER,
    'lr0': LR0, 'lrf': LRF, 'weight_decay': WEIGHT_DECAY, 'cos_lr': COS_LR,
    'patience': PATIENCE, 'save': True, 'save_period': SAVE_PERIOD,
    'project': str(OUTPUT_ROOT.resolve()), 'name': RUN_NAME, 'exist_ok': True,
}
print(json.dumps(to_jsonable(train_kwargs), indent=2))
train_result = model.train(**train_kwargs)
run_dir = Path(getattr(train_result, 'save_dir', OUTPUT_ROOT / RUN_NAME))
print('Run dir:', run_dir)

Using local dataset: C:\BME\MSC\4. félév\GL\GLHF\datasets\classification_cls
Dataset dir: C:\BME\MSC\4. félév\GL\GLHF\datasets\classification_cls
{
  "data": "C:\\BME\\MSC\\4. f\u00e9l\u00e9v\\GL\\GLHF\\datasets\\classification_cls",
  "task": "classify",
  "epochs": 100,
  "imgsz": 224,
  "batch": 32,
  "workers": 8,
  "device": 0,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "cos_lr": true,
  "patience": 20,
  "save": true,
  "save_period": 5,
  "project": "C:\\BME\\MSC\\4. f\u00e9l\u00e9v\\GL\\GLHF\\runs\\yolo26s_cls_roboflow",
  "name": "train_20260516_224211",
  "exist_ok": true
}
Ultralytics 8.4.51  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=fli

In [25]:
val_result = model.val(data=str(dataset_dir), task='classify', split='val')
metrics_dir = export_metrics(run_dir, train_result, val_result)
print('Metrics folder:', metrics_dir)
print('Best weights:', run_dir / 'weights' / 'best.pt')
print('Last weights:', run_dir / 'weights' / 'last.pt')

Ultralytics 8.4.51  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26s-cls summary (fused): 47 layers, 5,631,402 parameters, 0 gradients, 12.1 GFLOPs
train: C:\BME\MSC\4. flv\GL\GLHF\datasets\classification_cls\train... found 34600 images in 154 classes  
ERROR val: C:\BME\MSC\4. flv\GL\GLHF\datasets\classification_cls\valid... found 1234 images in 112 classes (requires 154 classes, not 112)
ERROR test: C:\BME\MSC\4. flv\GL\GLHF\datasets\classification_cls\test... found 548 images in 79 classes (requires 154 classes, not 79)
val: Fast image access  (ping: 0.00.0 ms, read: 517.5208.1 MB/s, size: 30.2 KB)
val: Scanning C:\BME\MSC\4. félév\GL\GLHF\datasets\classification_cls\valid... 1234 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1234/1234  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 78/78 72.4it/s 1.1s0.1s
                   all    0.00162      0.134
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.0ms postprocess per

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
